# Run LLM variants experiments

* Here, we run four variants including the embedding approach for comparison
    1. Use LLM-Embedding model, use LR on top of embedding [llm_encoder] -> implemented and reproduces EHRShot results
    2. Use LLM-Embedding model with linear head and fine-tuning [llm_encoder_ft]
    3. Use prompted LLM with text decoder directly as prediction [llm_decoder]
    4. Use prompted LLM with text decoder and fine-tuning [llm_decoder_ft]

In [ ]:
# Dependencies and definitions
import pickle
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Tuple, Protocol, List, Dict, Any

from serialization.text_encoder import (
    TextEncoder,
    Qwen3Embedding_8B_Encoder,
    Qwen3Embedding_4B_Encoder,
    Qwen3Embedding_0_6B_Encoder,
)

# Reuse the original evaluation routine (scaler, LR tuning, metrics, CIs) from 7_eval.py
import importlib.util
_EVAL_PATH = "/home/sthe14/ehrshot-benchmark/ehrshot/7_eval.py"
_spec = importlib.util.spec_from_file_location("ehrshot_eval", _EVAL_PATH)
_ehrshot_eval = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ehrshot_eval)  # type: ignore
run_evaluation_lr = _ehrshot_eval.run_evaluation


In [ ]:
# Run settings
tasks = ['guo_los', 'guo_readmission', 'guo_icu',
         'lab_thrombocytopenia', 'lab_hyperkalemia', 'lab_hypoglycemia', 'lab_hyponatremia', 'lab_anemia',
         'new_hypertension', 'new_hyperlipidemia', 'new_pancan', 'new_celiac', 'new_lupus', 'new_acutemi',
         'chexpert_Lung Lesion', 'chexpert_Pneumothorax', 'chexpert_Fracture', 'chexpert_Consolidation', 'chexpert_Cardiomegaly', 'chexpert_Enlarged Cardiomediastinum', 'chexpert_Edema', 'chexpert_Pneumonia', 'chexpert_Pleural Other', 'chexpert_Lung Opacity', 'chexpert_Atelectasis', 'chexpert_Pleural Effusion', 'chexpert_No Finding', 'chexpert_Support Devices']
ks = [8]  # [1, 2, 4, 8, 12, 16, 24, 32, 48, 64, 128, -1]
replicates = [0]  # [0, 1, 2, 3, 4]
NUM_THREADS = 40
PATH_TO_OUTPUT_FILE = '/home/sthe14/ehrshot-benchmark/EHRSHOT_ASSETS/experiments/llm_variants/llm_variants_results.csv'


In [ ]:
splits_to_serializations_path = '/home/sthe14/ehrshot-benchmark/EHRSHOT_ASSETS/benchmark/ehrshot_splits_to_serializations.csv'
tasks_serializations_path = '/home/sthe14/ehrshot-benchmark/EHRSHOT_ASSETS/benchmark/tasks_serializations.pkl'

# Load splits to serialization files with correct data types per column
dtype_dict = {'task': str, 'split_name': str, 'shot_size': int, 'fold': int, 'patient_id': int,
              'prediction_time': str, 'label_type': str, 'label_value': str, 'serialization_idx': int}
# ignore type error
splits_to_serializations = pd.read_csv(splits_to_serializations_path, dtype=dtype_dict, parse_dates=['prediction_time'])  # type: ignore
splits_to_serializations['label_value'] = splits_to_serializations['label_value'].apply(lambda x: x == 'True')

# Load the task to serialization files
with open(tasks_serializations_path, 'rb') as f:
    tasks_serializations = pickle.load(f)

# print(splits_to_serializations.columns)
# Select task chexpert_Edeme and split_name test all examples
# selected = splits_to_serializations[(splits_to_serializations['task'] == 'chexpert') & (splits_to_serializations['split_name'] == 'test')]
# print(selected)

In [ ]:
class llm_classifier(Protocol):
    """
    Interface for all LLM variants (llm_encoder, llm_encoder_ft, llm_decoder, llm_decoder_ft).

    Each implementing class must provide:
      run_evaluation(sub_task, X_train_texts, X_val_texts, X_test_texts, y_train, y_val, y_test, ...)
    returning: (best_model, scores_dict) where scores_dict matches 7_eval.py output.
    """
    def run_evaluation(
        self,
        sub_task: str,
        X_train_texts: list[str],
        X_val_texts: list[str],
        X_test_texts: list[str],
        y_train: np.ndarray,
        y_val: np.ndarray,
        y_test: np.ndarray,
        n_jobs: int = 1,
        test_patient_ids: np.ndarray | None = None,
        **kwargs,
    ) -> Tuple[object, dict]:
        ...


In [ ]:
class llm_encoder:
    """
    Concrete implementation of llm_classifier:
      - Uses Qwen3-Embedding backbone (size selectable; default 8B)
      - Prepends subtask-specific instruction to each serialization
      - Produces embeddings (last-token pool + L2 normalize as in text_encoder.py)
      - Calls 7_eval.py::run_evaluation() with 'lr_<solver>' for identical metrics/CI
    """
    _INSTR_PATH = "/home/sthe14/ehrshot-benchmark/ehrshot/serialization/task_to_instructions.json"

    def __init__(self, model_size: str = "8B", max_input_length: int = 4096):
        self.model_size = model_size
        self.max_input_length = max_input_length
        self._instructions = self._load_instructions()

        # Select Qwen3 backbone
        if model_size == "8B":
            self._backbone = Qwen3Embedding_8B_Encoder(max_input_length=max_input_length)
        elif model_size == "4B":
            self._backbone = Qwen3Embedding_4B_Encoder(max_input_length=max_input_length)
        elif model_size in {"0.6B", "0_6B", "0.6b"}:
            self._backbone = Qwen3Embedding_0_6B_Encoder(max_input_length=max_input_length)
        else:
            raise ValueError(f"Unsupported Qwen3 size: {model_size}. Use '8B' (default), '4B', or '0.6B'.")

        self._encoder = TextEncoder(self._backbone)

    def _load_instructions(self) -> dict:
        path = Path(self._INSTR_PATH)
        if not path.exists():
            raise FileNotFoundError(f"Instruction file not found: {self._INSTR_PATH}")
        with open(path, "r") as f:
            return json.load(f)

    def _instruction_for(self, sub_task: str) -> str:
        instruction_prefix = self._instructions.get("instruction_prefix", "")
        instruction = self._instructions.get(sub_task, "")
        if instruction_prefix != "":
            instruction = f"{instruction_prefix} {instruction}"
        assert isinstance(instruction, str), f"Instruction for task {sub_task} must be a string"
        return instruction

    def _encode_texts_with_instruction(self, texts: list[str], sub_task: str) -> np.ndarray:
        instr = self._instruction_for(sub_task)
        instructions = [instr] * len(texts)
        # No caching per requirements
        return self._encoder.encode_texts(instructions=instructions, texts=texts, cache_dir=None)

    # --- interface method ---
    def run_evaluation(
        self,
        sub_task: str,
        X_train_texts: list[str],
        X_val_texts: list[str],
        X_test_texts: list[str],
        y_train: np.ndarray,
        y_val: np.ndarray,
        y_test: np.ndarray,
        n_jobs: int = 1,
        test_patient_ids: np.ndarray | None = None,
        *,
        lr_solver: str = "lbfgs",
    ) -> Tuple[object, dict]:

        # 1) Build embeddings
        X_train = self._encode_texts_with_instruction(X_train_texts, sub_task=sub_task)
        X_val   = self._encode_texts_with_instruction(X_val_texts,   sub_task=sub_task)
        X_test  = self._encode_texts_with_instruction(X_test_texts,  sub_task=sub_task)

        # 2) Reuse original training/metrics/CI from 7_eval.py
        model_head = f"lr_{lr_solver}"  # matches 7_eval.py convention
        best_model, scores = run_evaluation_lr(
            X_train=X_train,
            X_val=X_val,
            X_test=X_test,
            y_train=y_train,
            y_val=y_val,
            y_test=y_test,
            model_head=model_head,
            n_jobs=n_jobs,
            test_patient_ids=test_patient_ids,
        )
        return best_model, scores

In [ ]:
# Set llm classifier setting
# Define llm_encoder class first from below

clf: llm_classifier = llm_encoder(model_size="0.6B", max_input_length=4096)
# For your results CSV schema
LABELING_FUNCTION = "llm_encoder"

In [ ]:
# Collect performance accuracy, f1, auroc, auprc
results: List[Dict[str, Any]] = []

# Routine from 7_eval.py, only consider llm setting
model = "llm"
for sub_task in tasks:
    # Debug: Select particular sub_task
    # if sub_task != 'guo_los':
    #     continue
    
    # For each k-shot sample we are evaluating...
    for k in ks:
        # For each replicate of this k-shot sample...
        for replicate in replicates:
            print(f"Model: {model} | Task: {sub_task} | k: {k} | replicate: {replicate}")
            
            task_split = splits_to_serializations[
                (splits_to_serializations['task'] == sub_task) &
                (splits_to_serializations['shot_size'] == k) &
                (splits_to_serializations['fold'] == replicate)
            ]
            train_split = task_split[task_split['split_name'] == 'train']
            val_split = task_split[task_split['split_name'] == 'val']
            # DEBUG: If using joint chexpert test set
            # test_task = sub_task if not sub_task.startswith('chexpert') else 'chexpert'
            test_split = splits_to_serializations[(splits_to_serializations['task'] == sub_task) & (splits_to_serializations['split_name'] == 'test')]
            
            X_train_k = [tasks_serializations[idx][1] for idx in train_split['serialization_idx'].values]
            X_val_k = [tasks_serializations[idx][1] for idx in val_split['serialization_idx'].values]
            X_test = [tasks_serializations[idx][1] for idx in test_split['serialization_idx'].values]
            y_train_k = np.array(train_split['label_value'].values)
            y_val_k = np.array(val_split['label_value'].values)
            y_test = np.array(test_split['label_value'].values)
            test_patient_ids = test_split['patient_id'].values
            
            # A bit different:
            # - test set already derived
            # - still need to do llm encoding here

            # Fit model with hyperparameter tuning
            # choose the concrete implementation you want to run

            best_model, scores = clf.run_evaluation(
                sub_task,
                X_train_k, X_val_k, X_test,
                y_train_k, y_val_k, y_test,
                n_jobs=NUM_THREADS,
                test_patient_ids=test_patient_ids,
            )

            # Save results
            for score_name, score_value in scores.items():
                results.append({
                    'labeling_function' : LABELING_FUNCTION,
                    'sub_task' : sub_task,
                    'model' : model,
                    'replicate' : replicate,
                    'k' : k,
                    'score' : score_name,
                    'value' : score_value['score'],
                    'std' : score_value['std'],
                    'lower' : score_value['lower'],
                    'mean' : score_value['mean'],
                    'upper' : score_value['upper'],
                })
            # Debug:
            print(f"Scores: {scores}")

print(f"Saving results to: {PATH_TO_OUTPUT_FILE}")
df: pd.DataFrame = pd.DataFrame(results)
# print(f"Added {df.shape[0] - (df_existing.shape[0] if df_existing is not None else 0)} rows")
df.to_csv(PATH_TO_OUTPUT_FILE)
print("Done!")

